# 05 — Calibration and portfolio evaluation

Compare all innovation models using the same realized test path. Calibration, sharpness, and proper scores must be interpreted together.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from innovcal.api.evaluation import compare_innovation_models

In [2]:
CACHE = ROOT / 'results/notebook_cache'
models = ('gaussian', 'student_t', 'bootstrap', 'diffusion')
forecasts = {}
for model in models:
    path = CACHE / f'forecast_{model}.npz'
    if path.exists():
        data = np.load(path)
        forecasts[model] = {key: data[key] for key in data.files}

assert forecasts, 'Run notebooks 03 and 04 first.'
y_true = next(iter(forecasts.values()))['y_true']
evaluation = compare_innovation_models(
    forecasts, y_true=y_true, dgp_name='financial', forecast_model='VAR'
)
evaluation.sort_values('energy_score')

,dgp,forecast_model,innovation_model,avg_coverage,avg_width,energy_score,crps,interval_score,ece,pit_deviation,...,width_1,coverage_2,width_2,coverage_3,width_3,coverage_4,width_4,nominal_coverage,coverage_error,abs_coverage_error
0,financial,VAR,gaussian,0.894628,0.037039,0.014032,0.006034,0.046202,0.014738,0.011157,...,0.025745,0.851240,0.033599,0.867769,0.035197,0.966942,0.053614,0.9,-0.005372,0.005372
2,financial,VAR,bootstrap,0.888430,0.036546,0.014072,0.006049,0.046443,0.023967,0.011405,...,0.024257,0.867769,0.033843,0.859504,0.036013,0.975207,0.052071,0.9,-0.011570,0.011570
1,financial,VAR,student_t,0.869835,0.035086,0.014126,0.006068,0.046231,0.043251,0.010496,...,0.024370,0.859504,0.032364,0.809917,0.033313,0.966942,0.050297,0.9,-0.030165,0.030165
3,financial,VAR,diffusion,0.944215,0.044211,0.014490,0.006182,0.048554,0.074518,0.021736,...,0.030279,0.933884,0.040678,0.933884,0.042172,0.991736,0.063714,0.9,0.044215,0.044215


In [3]:
rows = []
for name, forecast in forecasts.items():
    paths = forecast['forecast_paths']
    weights = np.full(paths.shape[-1], 1 / paths.shape[-1])
    portfolio_paths = paths @ weights
    realized = y_true @ weights
    q05 = np.quantile(portfolio_paths, 0.05, axis=0)
    rows.append({
        'innovation_model': name,
        'portfolio_95_var_exceedance': np.mean(realized < q05),
        'mean_portfolio_q05': q05.mean(),
    })
portfolio_evaluation = pd.DataFrame(rows)
portfolio_evaluation

,innovation_model,portfolio_95_var_exceedance,mean_portfolio_q05
0,gaussian,0.049587,-0.013195
1,student_t,0.049587,-0.012521
2,bootstrap,0.057851,-0.012364
3,diffusion,0.057851,-0.012159


In [4]:
evaluation.to_csv(CACHE / 'combined_evaluation.csv', index=False)
portfolio_evaluation.to_csv(CACHE / 'portfolio_evaluation.csv', index=False)